# Calibration & Cost-Sensitive Threshold

**Upstream:** Champion GBM (`best_model_hparam_gbm_default.pkl`)
**Goal:** Improve probability reliability and choose a decision threshold from business costs

---

### Purpose

Raw classifier scores rank applicants well but may be poorly calibrated as probabilities. This notebook applies **isotonic / sigmoid calibration**, compares Brier scores, and selects thresholds under explicit false-positive and false-negative cost assumptions.

### Steps

1. Reload Standard features and the train / validation split
2. Fit base GBM and calibrated variants
3. Compare Brier score and calibration curves
4. Cost-sensitive threshold grid (FN cost vs FP cost)
5. Persist calibrated model and recommended thresholds


## 0. Setup

Load the Standard feature matrix, recreate the stratified split, and prepare calibration utilities.


In [ ]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_score, recall_score, f1_score, confusion_matrix,
)

FEAT = Path("features")
OUT = Path("model_outputs")
OUT.mkdir(exist_ok=True)
RS = 42

feat = pd.read_csv(FEAT / "credit_train_features.csv")
with open(FEAT / "selected_features_standard.json") as f:
    sel = json.load(f)
cols = [c for c in sel["features"] if c in feat.columns]
X = feat[cols]
y = feat["Default"]
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RS)
print(X_tr.shape, X_va.shape, "default%", round(y.mean()*100, 2))


## 1. Base model and calibrated variants

Fit the champion Gradient Boosting pipeline, then wrap it with isotonic and sigmoid calibration using cross-validation on the training fold.


In [ ]:
def make_gbm():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", GradientBoostingClassifier(
            n_estimators=80, max_depth=4, learning_rate=0.1, subsample=0.8, random_state=RS,
        )),
    ])

base = make_gbm()
base.fit(X_tr, y_tr)
proba_base = base.predict_proba(X_va)[:, 1]

# Calibrate on training via CV (avoids using val twice for fit+eval of calibration)
cal_iso = CalibratedClassifierCV(make_gbm(), method="isotonic", cv=3)
cal_sig = CalibratedClassifierCV(make_gbm(), method="sigmoid", cv=3)
cal_iso.fit(X_tr, y_tr)
cal_sig.fit(X_tr, y_tr)
proba_iso = cal_iso.predict_proba(X_va)[:, 1]
proba_sig = cal_sig.predict_proba(X_va)[:, 1]

rows = []
for name, p in [("base", proba_base), ("isotonic", proba_iso), ("sigmoid", proba_sig)]:
    rows.append({
        "model": name,
        "roc_auc": roc_auc_score(y_va, p),
        "pr_auc": average_precision_score(y_va, p),
        "brier": brier_score_loss(y_va, p),
    })
cal_df = pd.DataFrame(rows).sort_values("brier")
print(cal_df.to_string(index=False))
best_cal_name = cal_df.iloc[0]["model"]
print("Best by Brier:", best_cal_name)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, p in [("base", proba_base), ("isotonic", proba_iso), ("sigmoid", proba_sig)]:
    frac, mean_p = calibration_curve(y_va, p, n_bins=10, strategy="quantile")
    ax.plot(mean_p, frac, marker="o", label=name)
ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="perfect")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of defaults")
ax.set_title("Calibration curves (validation)")
ax.legend()
plt.tight_layout()
plt.savefig(OUT / "plots/calibration_comparison.png", dpi=120, bbox_inches="tight")
plt.show()


## 2. Cost-sensitive threshold

Define:
- **FP cost** = cost of flagging a good loan (review / lost goodwill)
- **FN cost** = cost of missing a default (loss given default proxy)

Expected cost at threshold \(t\):
`FP_cost × false_positives + FN_cost × false_negatives`

Default scenario below: FN 5× more expensive than FP (typical when loss >> review cost). Change the ratio to match your book.


In [ ]:
def cost_curve(y_true, proba, fp_cost=1.0, fn_cost=5.0):
    rows = []
    for t in np.linspace(0.05, 0.95, 37):
        pred = (proba >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
        cost = fp_cost * fp + fn_cost * fn
        rows.append({
            "threshold": t, "cost": cost, "fp": int(fp), "fn": int(fn),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0),
            "flag_rate": pred.mean(),
        })
    return pd.DataFrame(rows)

# Use best-calibrated probabilities for thresholding
proba_use = {"base": proba_base, "isotonic": proba_iso, "sigmoid": proba_sig}[best_cal_name]

scenarios = {
    "fn5_fp1": (1.0, 5.0),
    "fn3_fp1": (1.0, 3.0),
    "fn1_fp1": (1.0, 1.0),
    "max_f1": None,
}
chosen = {}
for name, costs in scenarios.items():
    if costs is None:
        # max F1
        best_t, best_f1 = 0.5, -1
        for t in np.linspace(0.05, 0.95, 37):
            f1 = f1_score(y_va, (proba_use >= t).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        chosen[name] = {"threshold": best_t, "rule": "max_f1"}
    else:
        curve = cost_curve(y_va, proba_use, *costs)
        i = curve["cost"].idxmin()
        chosen[name] = {**curve.loc[i].to_dict(), "rule": f"min_cost fp={costs[0]} fn={costs[1]}"}

print("Recommended thresholds:")
for k, v in chosen.items():
    print(f"  {k}: thr={v['threshold']:.2f}  {v.get('rule')}")

# Primary recommendation: fn5_fp1 if costs unknown but defaults are costly
primary = chosen["fn5_fp1"]
print(f"\n>>> Primary cost-based threshold: {primary['threshold']:.2f}")


In [ ]:
curve = cost_curve(y_va, proba_use, 1.0, 5.0)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(curve["threshold"], curve["cost"], label="Expected cost (FP=1, FN=5)")
ax.axvline(primary["threshold"], color="red", ls="--", label=f"min cost @ {primary['threshold']:.2f}")
ax.set_xlabel("Threshold"); ax.set_ylabel("Cost"); ax.legend()
ax.set_title("Cost vs threshold")
plt.tight_layout()
plt.savefig(OUT / "plots/cost_threshold.png", dpi=120, bbox_inches="tight")
plt.show()
print(curve.loc[curve["cost"].idxmin()])


## 3. Persist calibrated champion and threshold

Refit the preferred calibrated model on all training data and save the pickle plus threshold metadata for production scoring.


In [ ]:
# Refit calibration on full training data for production
cal_final = CalibratedClassifierCV(make_gbm(), method="isotonic" if best_cal_name=="isotonic" else "sigmoid", cv=3)
if best_cal_name == "base":
    cal_final = make_gbm()
cal_final.fit(X, y)

with open(OUT / "best_model_calibrated.pkl", "wb") as f:
    pickle.dump(cal_final, f)

meta = {
    "base_model": "gradient_boosting",
    "calibration": best_cal_name,
    "threshold_primary": float(primary["threshold"]),
    "threshold_max_f1": float(chosen["max_f1"]["threshold"]),
    "threshold_scenarios": {k: {kk: (float(vv) if isinstance(vv, (float, np.floating, int, np.integer)) else vv)
                                for kk, vv in v.items()} for k, v in chosen.items()},
    "val_calibration_metrics": cal_df.to_dict(orient="records"),
    "features": cols,
    "fp_cost_default": 1.0,
    "fn_cost_default": 5.0,
}
with open(OUT / "calibration_threshold_meta.json", "w") as f:
    json.dump(meta, f, indent=2, default=float)

cal_df.to_csv(OUT / "calibration_metrics.csv", index=False)
curve.to_csv(OUT / "cost_curve_fn5_fp1.csv", index=False)
print("Saved calibrated model + meta to", OUT.resolve())
print(json.dumps(meta, indent=2, default=float)[:800])
